<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase8-multi-agent-governance/03_immutable_audit_logging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 8: Immutable Audit Logging
**Goal**: Build a tamper-evident audit log that captures
      every governance event across the multi-agent
      pipeline with cryptographic integrity verification.
Regulatory mapping: EU AI Act Article 12 record-keeping,
                    FRIA accountability requirements.
**Date**: June 2026.
**Status**: In Progress

In [12]:
import time
import json
import hashlib
import pandas as pd
from datetime import datetime
from google import genai
from google.colab import userdata, drive
import os

# Setup
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)

# - IMMUTABLE AUDIT LOGGER -
# Each log entry is cryptographically chained to the previous entry.
# Any tampering with a past entry breaks the chain and is detectable.

class ImmutableAuditLog:
  """
  A tamper-evident audit log using hash chaining.
  Each entry contains the hash of the previous entry.
  Any modification to any entry breaks the chain.
  This makes retrospective tampering detectable.
  """
  def __init__(self):
    self.entries = []
    self.log_file = SAVE_PATH + "immutable_audit_log.json"

  def _hash_entry(self, entry):
    """Generate SHA-256 hash of a log entry."""
    entry_string = json.dumps(entry, sort_keys=True)
    return hashlib.sha256(
        entry_string.encode()
    ).hexdigest()

  def _previous_hash(self):
    """Get the hash of the previous entry in the chain."""
    if not self.entries:
      return "GENESIS"
    return self.entries[-1]["entry_hash"]

  def log(self, event_type, agent, details,
          authorized_by=None):
    """Add a tamper-evident entry to the log."""
    # This method is currently empty. It might be intended to call add_entry.
    # For now, it just passes.
    pass

  def add_entry(self, event_type, agent, details, authorized_by=None):
    """Add a new entry to the log."""
    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S")

    entry = {
        "entry_number": len(self.entries) + 1,
        "timestamp": timestamp,
        "event_type": event_type,
        "agent": agent,
        "details": details,
        "authorized_by": authorized_by,
        "previous_hash": self._previous_hash(),
    }

    entry["entry_hash"] = self._hash_entry(entry)
    self.entries.append(entry)

    print(f"  LOG [{entry['entry_number']}]"
          f"{event_type}: {agent}")
    return entry

  def verify_integrity(self):
    """
    Verify the integrity of the log entries.
    Detect any tampering with past entries.
    """
    print("\n====== AUDIT LOG INTEFRITY CHECK ======")

    if not self.entries:
      print("Log is empty.")
      return True

    previous_hash = "GENESIS"
    all_valid     = True

    for entry in self.entries:
      stored_hash = entry["entry_hash"]
      stored_prev = entry["previous_hash"]

      # Rebuild the entry without hash to verify
      entry_to_verify = {k: v for k, v in entry.items()
                         if k != "entry_hash"}
      computed_hash  = self._hash_entry(entry_to_verify)

      chain_valid = stored_prev == previous_hash
      hash_valid  = stored_hash == computed_hash

      status = "VALID" if (chain_valid and hash_valid) \
               else "TAMPERED"

      if status == "TAMPERED":
        all_valid = False

      print(f" Entry {entry['entry_number']:02d}: "
            f"{status} / "
            f"{entry['event_type']:25} / "
            f"Hash: {stored_hash[:12]}...")

      previous_hash = stored_hash

    print(f"\nOverall integrity: "
          f"{'VERIFIED' if all_valid else 'COMPROMISED'}")
    return all_valid

  def save(self):
    """Save the audit log to Google Drive."""
    with open(self.log_file, "w") as f:
      json.dump(self.entries, f, indent=2)
    print(f"\nAudit log saved to {self.log_file}")

  def display(self):
    """Display the full audit log in a Pandas DataFrame."""
    if not self.entries:
      print("Log is empty.")
      return

    df = pd.DataFrame([{
            "entry":        e["entry_number"],
            "timestamp":    e["timestamp"],
            "event_type":   e["event_type"],
            "agent":        e["agent"],
            "authorized_by": e["authorized_by"] or "System",
            "hash":         e["entry_hash"][:12] + "...",
        } for e in self.entries])
    return df

print("====== IMMUTABLE AUDIT LOG READY ======")
print("Hash algorithm: SHA-256")
print("Chain method:   each entry hashes previous")
print("Tamper detection: any modification breaks chain")
print("\nImmutable logger ready ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
====== IMMUTABLE AUDIT LOG READY ======
Hash algorithm: SHA-256
Chain method:   each entry hashes previous
Tamper detection: any modification breaks chain

Immutable logger ready ✅


In [17]:
# - SYSTEMS AND AGENTS (same as previous lesson 1 and 2) -
SYSTEM_UNDER_AUDIT = {
    "ststem_name":    "TalentMatch AI",
    "purpose":        "Automated CV screening",
    "sector":         "employment",
    "known_metrics":  {
        "gender_disparate_impact":    0.43,
        "age_disparate_impact":       0.20,
        "ethnicity_disparate_impact": 0.80,
    },
    "human_oversight":  "None currently implemented",
    "documentation":    "Partial. No FRIA completed.",
}

def bias_audit_agent(system):
  metrics = system["known_metrics"]
  findings = []
  critical = False
  for dimension, ratio in metrics.items():
    if ratio < 0.80:
      severity = "CRITICAL" if ratio < 0.50 else "HIGH"
      if severity == "CRITICAL":
        critical = True
      findings.append({
          "dimension": dimension,
          "ratio":     ratio,
          "severity":  severity,
          "compliant": False})
    else:
      findings.append({
          "dimension": dimension,
          "ratio":     ratio,
          "severity":  "PASS",
          "compliant": True})

  return {
      "agent":          "Bias Audit Agent",
      "article":        "EU AI Act Article 10",
      "findings":        findings,
      "critical_breach": critical,
      "verdict":         "FAIL" if not all(
                        f["compliant"] for f in findings)
                      else "PASS",
  }

def oversight_audit_agent(system):
  oversight     = system["human_oversight"].lower()
  has_oversight = "none" not in oversight
  return {
      "agent":          "Oversight Audit Agent",
      "article":        "EU AI Act Article 14",
      "findings":        system["human_oversight"],
      "critical_breach": not has_oversight,
      "verdict":         "PASS" if has_oversight
                            else "FAIL"
  }

def documentation_audit_agent(system):
  docs =     system["documentation"].lower()
  complete = "partial" not in docs and "no " not in docs
  return {
      "agent":          "Documentation Audit Agent",
      "article":        "EU AI Act Article 11 and 18",
      "findings":        system["documentation"],
      "critical_breach": False,
      "verdict":         "PASS" if complete
                            else "FAIL"
  }

AUDIT_AGENTS = {
    "bias":      bias_audit_agent,
    "oversight": oversight_audit_agent,
    "documentation": documentation_audit_agent,
}

# KILL-SWITCH CLASS -
class killSwitch:
  def __init__(self):
    self.triggered      = False
    self.trigger_reason = None
    self.trigger_agent  = None
    self.trigger_time   = None
    self.authorized_by  = None

  def trigger(self, reason, agent):
    self.triggered      = True
    self.trigger_reason = reason
    self.trigger_agent  = agent
    self.trigger_time   = datetime.now().strftime(
                          "%Y-%m-%d %H:%M:%S")

  def reset(self, authorized_by):
    if not authorized_by:
      return False
    self.triggered      = False
    self.authorized_by  = authorized_by
    return True

  def status(self):
    return "HALTED" if self.triggered else "OPERATIONAL"

# - FULLY LOGGED GOVERNANCE PIPELINE -
def logged_governance_pipeline(system, agents, audit_log):
  """
  complete nulti-agent pipeline with immutable logging.
  Every governance event is captured and chained.
  """
  print("=" * 60)
  print(f"LOGGED AUDIT: {system['ststem_name']}")
  print("=" * 60)

  ks = killSwitch()

  # Log audit start
  audit_log.add_entry(
      event_type = "AUDIT_STARTED",
      agent       = "Governance Controller",
      details    = f"Auditing {system['ststem_name']}." # Added closing quote here
                   f"Sector: {system['sector']}.",
      authorized_by = "Governance Controller"
  )

  agents_run    = 0
  agents_halted = 0

  for agent_name, agent_function in agents.items():

    if ks.triggered:
      audit_log.add_entry(
          event_type = "AGENT_SKIPPED",
          agent      = agent_name,
          details    = "Skipped due to active kill-switch."
      )
      agents_halted += 1
      continue

    # Log agent start
    audit_log.add_entry(
        event_type = "AGENT_STARTED",
        agent      = agent_name,
        details    = f"Running {agent_name} audit agent."
    )

    result = agent_function(system)
    agents_run += 1

    # Log agent result
    audit_log.add_entry(
        event_type  = "AGENT_COMPLETED",
        agent       = result["agent"],
        details     = f"Verdict: {result['verdict']}."
                      f"Article: {result['article']}.",
    )

    print(f"\n{agent_name} agent: {result['verdict']}")

    if result.get("critical_breach"):
      ks.trigger(
          reason = f"Critical breach: {result['article']}",
          agent  = result["agent"]
      )

      # Log kill-switch trigger
      audit_log.add_entry(
          event_type  = "KILL_SWITCH_TRIGGERED",
          agent       = result["agent"],
          details     = f"Critical breach detected. "
                        f"{result['article']}."
                        f"System halted immediately."
      )

      print(f"\nKILL-SWITCH TRIGGERED: {result['agent']}")
      print(f"System halted. Requesting authorization...")

      # Simulate human authorization for demo
      # In production this would use input()
      authorized_by = (
          "Steve Onyeke, Lead AI Auditor, Afrispan Data Labs"
      )
      position =(
          "Critical breach confirmed."
          "Audit to continue under supervision "
          "pending full remediation plan."
      )

      ks.reset(authorized_by=authorized_by)

      # Log human authorization
      audit_log.add_entry(
          event_type  = "HUMAN_AUTHORIZATION",
          agent       = "Human Auditor",
          details     = f"position: {position}",
          authorized_by = authorized_by
      )

      print(f"Authorized_by: {authorized_by}")

  #log audit completion
  critical_breaches = sum(
    1 for e in audit_log.entries
    if e["event_type"] == "KILL_SWITCH_TRIGGERED"
)
  overall = "CRITICAL NON-COMPLIANCE" if critical_breaches > 0 \
            else "COMPLIANT"
  audit_log.add_entry(
      event_type = "AUDIT_COMPLETED",
      agent      = "Governance Controller",
      details    = f"Audit completed. "
                   f"Agents run: {agents_run}. "
                   f"Agents halted: {agents_halted}. "
                   f"Overall: {overall}."
  )

  return {
      "agents_run":    agents_run,
      "agents_halted": agents_halted,
      "verdict":       overall

  }

# - RUN THE FULLY LOGGED PIPELINE -
print("====== IMMUTABLE AUDIT LOG DEMONSTRATION ======\n")
print("All governance events will be logged and chained.\n")

audit_log = ImmutableAuditLog()
result = logged_governance_pipeline(
         SYSTEM_UNDER_AUDIT,AUDIT_AGENTS, audit_log)

print(f"\nAudit verdict: {result['verdict']}")
print(f"Agents run: {result['agents_run']}")
print(f"Agents halted: {result['agents_halted']}")

====== IMMUTABLE AUDIT LOG DEMONSTRATION ======

All governance events will be logged and chained.

LOGGED AUDIT: TalentMatch AI
  LOG [1]AUDIT_STARTED: Governance Controller
  LOG [2]AGENT_STARTED: bias
  LOG [3]AGENT_COMPLETED: Bias Audit Agent

bias agent: FAIL
  LOG [4]KILL_SWITCH_TRIGGERED: Bias Audit Agent

KILL-SWITCH TRIGGERED: Bias Audit Agent
System halted. Requesting authorization...
  LOG [5]HUMAN_AUTHORIZATION: Human Auditor
Authorized_by: Steve Onyeke, Lead AI Auditor, Afrispan Data Labs
  LOG [6]AGENT_STARTED: oversight
  LOG [7]AGENT_COMPLETED: Oversight Audit Agent

oversight agent: FAIL
  LOG [8]KILL_SWITCH_TRIGGERED: Oversight Audit Agent

KILL-SWITCH TRIGGERED: Oversight Audit Agent
System halted. Requesting authorization...
  LOG [9]HUMAN_AUTHORIZATION: Human Auditor
Authorized_by: Steve Onyeke, Lead AI Auditor, Afrispan Data Labs
  LOG [10]AGENT_STARTED: documentation
  LOG [11]AGENT_COMPLETED: Documentation Audit Agent

documentation agent: FAIL
  LOG [12]AUDIT_C

In [18]:
# ── INTEGRITY VERIFICATION ──
print("====== STEP 1: Verify untampered log ======")
audit_log.verify_integrity()

# ── DISPLAY FULL LOG ──
print("\n====== COMPLETE AUDIT LOG ======")
df = audit_log.display()
print(df.to_string(index=False))

# ── DEMONSTRATE TAMPER DETECTION ──
print("\n====== STEP 2: Simulate tampering ======")
print("Modifying entry 4 to hide the kill-switch event...")
original_details = audit_log.entries[3]["details"]
audit_log.entries[3]["details"] = "Verdict: PASS. All clear."
print(f"Original: {original_details}")
print(f"Modified: {audit_log.entries[3]['details']}")

print("\n====== STEP 3: Verify tampered log ======")
audit_log.verify_integrity()

# ── RESTORE AND SAVE ──
print("\n====== STEP 4: Restore and save ======")
audit_log.entries[3]["details"] = original_details
audit_log.entries[3]["entry_hash"] = audit_log._hash_entry(
    {k: v for k, v in audit_log.entries[3].items()
     if k != "entry_hash"}
)
audit_log.verify_integrity()
audit_log.save()

print("\nImmutable audit log demonstration complete ✅")

====== STEP 1: Verify untampered log ======

====== AUDIT LOG INTEFRITY CHECK ======
 Entry 01: VALID / AUDIT_STARTED             / Hash: b3d6683215a6...
 Entry 02: VALID / AGENT_STARTED             / Hash: 073dc80ad6b6...
 Entry 03: VALID / AGENT_COMPLETED           / Hash: 8bae181de688...
 Entry 04: VALID / KILL_SWITCH_TRIGGERED     / Hash: 10d1c6851691...
 Entry 05: VALID / HUMAN_AUTHORIZATION       / Hash: 4eba89c9b220...
 Entry 06: VALID / AGENT_STARTED             / Hash: 2470f2d18845...
 Entry 07: VALID / AGENT_COMPLETED           / Hash: 5e6a5f7b4059...
 Entry 08: VALID / KILL_SWITCH_TRIGGERED     / Hash: 18e78f8f0125...
 Entry 09: VALID / HUMAN_AUTHORIZATION       / Hash: 079c2dff6e18...
 Entry 10: VALID / AGENT_STARTED             / Hash: 8a7d7e0ae115...
 Entry 11: VALID / AGENT_COMPLETED           / Hash: 326a16765773...
 Entry 12: VALID / AUDIT_COMPLETED           / Hash: 7de55fa4f7f3...

Overall integrity: VERIFIED

====== COMPLETE AUDIT LOG ======
 entry           timesta